# Interactive LC optics — PyPI 0.3.0
Choose a **CPU** runtime, then **Runtime → Run all** to prepare the interface. Setup installs the pinned production package and verifies its runtime files. Only the interface's **Run** button starts a scientific calculation. No uploads or external helper scripts are needed.

Colab may warn that this notebook was not authored by Google. This is the standard third-party notebook warning, not a scientific validation failure. Review the source and trust its publisher before allowing execution; notebook code can access its runtime and any storage you explicitly connect. Setup does not mount Drive.

The advanced cell is collapsed where Colab supports form metadata. Ordinary Jupyter may show it. Both setup cells must execute in order. Rerunning preserves the existing interface and completed snapshot; a failed setup never replaces it.


In [ ]:
#@title Advanced setup implementation (no scientific execution)
def prepare_pypi_interface(previous=None):
    import hashlib, importlib, importlib.metadata as md, io, json
    import subprocess, sys, urllib.request, zipfile
    from pathlib import Path
    version = "0.3.0"
    wheel_hash = "9ab34f4ec244716b3e6f0431e4cac1e0d8a1ebeb17a7064c6422f1ed6b4e3d60"
    print("Checking LCProp 0.3.0…")
    loaded = any(n == "lcprop" or n.startswith("lcprop.") for n in sys.modules)
    try:
        installed = md.version("lcprop")
    except md.PackageNotFoundError:
        installed = None
    if loaded and installed != version:
        raise RuntimeError("A different LCProp is loaded. Restart the runtime before setup.")
    if installed != version:
        if previous is not None:
            raise RuntimeError("Existing interface has a different installation; preserve exports and restart.")
        print("Installing the pinned production package…")
        result = subprocess.run([sys.executable, "-m", "pip", "--isolated", "install",
            "--index-url", "https://pypi.org/simple", "lcprop[colab]==0.3.0"],
            capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("Installation failed. Check network/runtime compatibility. " + result.stderr[-2000:])
        importlib.invalidate_caches()
    # Reuse compatible loaded numerical dependencies; never upgrade them explicitly.
    for name in ("numpy", "scipy", "matplotlib", "ipywidgets", "IPython"):
        try:
            module = importlib.import_module(name)
        except ImportError as error:
            raise RuntimeError('Missing notebook dependencies: install "lcprop[colab]==0.3.0" in this kernel, then retry.') from error
        if getattr(module, "__version__", md.version(name)) != md.version(name):
            raise RuntimeError(f"Loaded {name} differs from installed metadata; restart the runtime.")
    with urllib.request.urlopen("https://pypi.org/pypi/lcprop/0.3.0/json", timeout=30) as response:
        metadata = json.load(response)
    wheel = next(f for f in metadata["urls"] if f["filename"] == "lcprop-0.3.0-py3-none-any.whl")
    if wheel["digests"]["sha256"] != wheel_hash or not wheel["url"].startswith("https://files.pythonhosted.org/"):
        raise RuntimeError("Published wheel identity mismatch; setup stopped.")
    with urllib.request.urlopen(wheel["url"], timeout=60) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != wheel_hash:
        raise RuntimeError("Downloaded wheel identity mismatch; setup stopped.")
    distribution = md.distribution("lcprop")
    if distribution.version != version:
        raise RuntimeError("Installed LCProp version mismatch.")
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        members = [n for n in archive.namelist() if n.startswith("lcprop/") and not n.endswith("/")]
        if len(members) != 267:
            raise RuntimeError("Unexpected runtime inventory.")
        for name in members:
            if Path(distribution.locate_file(name)).read_bytes() != archive.read(name):
                raise RuntimeError(f"Installed package identity mismatch: {name}")
    package_root = Path(distribution.locate_file("lcprop")).resolve()
    for name, module in tuple(sys.modules.items()):
        if name == "lcprop" or name.startswith("lcprop."):
            path = getattr(module, "__file__", None)
            if path and not Path(path).resolve().is_relative_to(package_root):
                raise RuntimeError("LCProp is imported from a checkout/other installation; restart in a clean kernel.")
    from lcprop.notebook.interactive_lc import InteractiveLC
    if previous is not None and not isinstance(previous, InteractiveLC):
        raise RuntimeError("Existing interface is incompatible; preserve exports and restart.")
    interface = previous if previous is not None else InteractiveLC()
    provenance = {"version": version, "wheel_sha256": wheel_hash,
                  "package_root": str(package_root), "runtime_files_verified": len(members),
                  "dependencies": {n: md.version(n) for n in ("numpy", "scipy", "matplotlib", "ipywidgets", "IPython")}}
    from IPython.display import display
    display(interface.widget)
    print("Ready — configure the experiment, then press Run. Setup did not run the solver.")
    return interface, provenance


In [ ]:
#@title Prepare LCProp interface
try:
    app, installation_provenance = prepare_pypi_interface(globals().get("app"))
    print(installation_provenance)
except Exception as error:
    print(f"Failed — {error}")
    raise


## Configure, Run, export
The packaged interface retains beam position/radii/power, applied voltage, Restore qualified preset, longitudinal intensity and director-reorientation slices, width curves, and 120/180/300-second budgets. Editing controls updates the launch preview only. Modified configurations are exploratory; convergence is not a stationary-soliton claim.

Plots and exports belong to the last successful immutable completed snapshot. Export before ending the session; Colab storage is temporary. Execution is synchronous, so there is no responsive Stop guarantee. Budget checks occur between completed slices. Plan at least 1 GiB available RAM and allow slower hosted CPU execution.

The setup uses production PyPI, verifies the qualified wheel hash and all 267 installed runtime files, and prints installation provenance. It never imports Qt, LaunchPlane or CuPy. If an incompatible package is already loaded, follow the restart diagnostic; no automatic restart is performed. A package already installed without notebook extras gets an actionable dependency message. This new notebook requires hosted Colab qualification; the package's prior qualification does not qualify this notebook automatically.
